In [13]:
# import pandas as pd
# import numpy as np

# # 1. Load Dataset Final
# df_final = pd.read_csv('../../datas/dataset_final.csv')
# print(f"Data Loaded: {df_final.shape}")

# print(df_final.info())

# # 2. Siapkan "Database Profil User"
# # Kita butuh daftar unik setiap user beserta atribut fisiknya untuk dicocokkan
# # Kita pakai kolom dengan akhiran _x karena itu biasanya data profil awal
# cols_profile = [
#     'User_ID', 'Age_x', 'Gender_x', 'Height_cm_x', 'Initial_Weight_kg_x', 
#     'Goal_x', 'Workout_Frequency_x', 'level_x', 'Environment'
# ]

# # Ambil 1 baris per user saja (Drop duplikat karena User_ID berulang di dataset_final)
# df_users_db = df_final[cols_profile].drop_duplicates(subset=['User_ID']).copy()

# print(f"Jumlah User Unik di Database: {len(df_users_db)}")

# # --- FUNGSI 1: CARI KEMBARAN (SIMILARITY) ---
# def find_similar_user(new_user_data):
#     """
#     Mencari User_ID dari database yang paling mirip dengan user baru.
#     """
#     candidates = df_users_db.copy()
    
#     # FILTER 1: Cari yang Goal-nya SAMA (Wajib)
#     # Kalau goal beda (misal user mau Muscle Gain tapi dikasih Weight Loss), jadwalnya pasti salah.
#     candidates = candidates[candidates['Goal_x'] == new_user_data['Goal']]
    
#     # FILTER 2: Cari yang Frekuensinya SAMA (Wajib)
#     # Biar jadwalnya pas (misal user minta 3 hari, dikasih jadwal 3 hari)
#     candidates = candidates[candidates['Workout_Frequency_x'] == new_user_data['Workout_Frequency'] & candidates['Workout_Frequency_x'] == new_user_data['Workout_Frequency']] 
    
#     # Jika tidak ada yang cocok persis, kita longgarkan filter (ambil Goal aja)
#     if candidates.empty:
#         print("Note: Tidak menemukan frekuensi yang sama persis, mencari berdasarkan Goal saja...")
#         candidates = df_users_db[df_users_db['Goal_x'] == new_user_data['Goal']].copy()
    
#     # FILTER 3: Hitung Skor Kemiripan (Similarity Score)
#     # Semakin kecil skor, semakin mirip.
#     # Kita bandingkan Umur, Tinggi, dan Berat.
    
#     # Kita ubah Gender jadi angka dulu (0/1) buat hitungan
#     gender_input = 1 if new_user_data['Gender'] == 'Male' else 0
#     candidates['Gender_Num'] = candidates['Gender_x'].apply(lambda x: 1 if x == 'Male' else 0)
    
#     candidates['Similarity_Score'] = (
#         abs(candidates['Age_x'] - new_user_data['Age']) * 1 +            # Bobot Umur: 1
#         abs(candidates['Height_cm_x'] - new_user_data['Height']) * 2 +   # Bobot Tinggi: 2
#         abs(candidates['Initial_Weight_kg_x'] - new_user_data['Weight']) * 2 + # Bobot Berat: 2
#         abs(candidates['Gender_Num'] - gender_input) * 50                # Bobot Gender: 50 (Harus sama!)
#     )
    
#     # Urutkan dari skor terkecil (paling mirip)
#     best_match = candidates.sort_values('Similarity_Score').iloc[0]
    
#     return best_match['User_ID'], best_match['Similarity_Score']

# # --- FUNGSI 2: AMBIL JADWAL ---
# def get_workout_plan(user_id_target):
#     """
#     Mengambil semua jadwal latihan milik User ID tertentu.
#     """
#     # Ambil baris data milik user tersebut
#     schedule = df_final[df_final['User_ID'] == user_id_target].copy()
    
#     # Pilih kolom yang mau ditampilkan sebagai jadwal
#     cols_display = [
#         'Day', 'Muscle Group', 'Exercise Name', 
#         'Equipment', 'Sets', 'Reps', 'Instructions'
#     ]
    
#     # Rapikan: Urutkan berdasarkan Hari, lalu Buang duplikat latihan (kalau ada)
#     # Kita asumsikan kolom 'Day' bisa diurutkan string-nya (Day 1, Day 2...)
#     schedule_clean = schedule[cols_display].drop_duplicates()
    
#     return schedule_clean

# # --- CONTOH PEMAKAIAN ---

# # 1. Data User Baru (Input dari Frontend/Aplikasi)
# new_user = {
#     'Age': 25,
#     'Gender': 'Male',
#     'Height': 175,
#     'Weight': 70,
#     'Goal': 'Muscle Gain',        # Harus sama persis tulisannya dengan di CSV
#     'Workout_Frequency': 4,       # Mau latihan 4 hari
#     'Level': 'Beginner',
#     'Environment': 'Home'
# }

# print("Mencari rekomendasi untuk user baru...")
# print(f"Profil: {new_user}")

# # 2. Cari ID User yang mau "dicontek"
# matched_id, score = find_similar_user(new_user)
# print(f"\n✅ Ditemukan Kembaran! User ID: {matched_id} (Skor Beda: {score})")
# print("Mengambil jadwal latihan dari User ID tersebut...\n")

# # 3. Tampilkan Jadwalnya
# rekomendasi_jadwal = get_workout_plan(matched_id)

# # Menampilkan per Hari biar rapi
# unique_days = rekomendasi_jadwal['Day'].unique()
# sorted_days = sorted(unique_days) # Mengurutkan Day 1, Day 2, dst

# for day in sorted_days:
#     print(f"📅 {day}")
#     day_plan = rekomendasi_jadwal[rekomendasi_jadwal['Day'] == day]
#     # Tampilkan kolom penting saja biar ga kepanjangan
#     display(day_plan[['Muscle Group', 'Exercise Name', 'Sets', 'Reps', 'Equipment']])
#     print("-" * 50)

In [14]:
import pandas as pd
import numpy as np
import pickle

In [15]:
path = '../../models/model_workout.pickle'

with open(path, 'rb') as f:
    model_data = pickle.load(f)
    
knn = model_data['knn_model']
scaler = model_data['scaler']
weights = model_data['weights'] # Load the weights we saved!
db_profiles = model_data['profiles_db']
db_schedule = model_data['schedule_db']
encoders = model_data['encoders']
feature_order = model_data['features']

print("✅ Model Loaded Successfully.")

def predict_workout_plan(user_input):
    print("\n" + "="*50)
    print(f"🔎 SEARCHING PLAN FOR: {user_input['Goal']} | {user_input['Frequency']} Days | {user_input['Environment']}")
    print("="*50)
    
    # --- A. ENCODE INPUT (Text -> Numbers) ---
    try:
        # We use [0] because transform expects a list
        goal_enc = encoders['goal'].transform([user_input['Goal']])[0]
        level_enc = encoders['level'].transform([user_input['Level']])[0]
        gender_enc = encoders['gender'].transform([user_input['Gender']])[0]
        env_enc = encoders['environment'].transform([user_input['Environment']])[0]
    except ValueError as e:
        print(f"❌ Error: Input contains unknown category (e.g., typo in Goal). Details: {e}")
        return

    # --- B. PREPARE DATAFRAME ---
    # Must match 'feature_order' exactly
    input_data = {
        'Goal_Encoded': goal_enc,
        'Workout_Frequency_x': user_input['Frequency'],
        'level_Encoded': level_enc,
        'Gender_Encoded': gender_enc,
        'Age_x': user_input['Age'],
        'Initial_Weight_kg_x': user_input['Weight'],
        'environment_Encoded': env_enc
    }
    
    # Create DF with exact column order
    input_df = pd.DataFrame([input_data])[feature_order]
    
    # --- C. SCALE ---
    input_scaled = scaler.transform(input_df)
    
    # --- D. APPLY SAVED WEIGHTS (CRITICAL STEP) ---
    # We must treat the new user exactly like we treated the training data
    input_weighted = pd.DataFrame(input_scaled, columns=feature_order)
    for col, weight in weights.items():
        input_weighted[col] = input_weighted[col] * weight
        
    # --- E. FIND NEIGHBOR ---
    # Find the single best match (n_neighbors=1) or top 3
    distances, indices = knn.kneighbors(input_weighted.values, n_neighbors=3)
    
    # Get the User ID of the match
    matched_index = indices[0][0]
    matched_user = db_profiles.iloc[matched_index]
    matched_user_id = matched_user['User_ID']
    
    print(f"✅ FOUND MATCH: User {matched_user_id}")
    print(f"   Match Goal : {matched_user['Goal_x']}")
    print(f"   Match Freq : {matched_user['Workout_Frequency_x']} Days")
    print(f"   Match Env  : {matched_user['Environment']}")
    print("-" * 50)
    
    # --- F. RETRIEVE SCHEDULE ---
    # Look up the actual exercises from the schedule database
    schedule = db_schedule[db_schedule['User_ID'] == matched_user_id].copy()
    
    # Sort by Day
    if 'Day' in schedule.columns:
        # Simple string sort "Day 1", "Day 2" usually works, 
        # but for robustness we extracts numbers if needed.
        schedule = schedule.sort_values('Day')

    # Display Columns
    cols_show = ['Day', 'Muscle Group', 'Exercise Name', 'Sets', 'Reps']
    
    # Group by Day for cleaner output
    unique_days = schedule['Day'].unique()
    
    for day in unique_days:
        print(f"\n📅 {day}")
        day_data = schedule[schedule['Day'] == day][cols_show]
        display(day_data.reset_index(drop=True))


# ==========================================
# 3. TEST RUN
# ==========================================
new_user_input = {
    'Age': 25,
    'Gender': 'Male',
    'Weight': 70,
    'Goal': 'Muscle Gain',       # Must match your dataset (Case Sensitive)
    'Frequency': 4,              # Must be integer
    'Level': 'Beginner',         # Must match dataset
    'Environment': 'Home'        # Must match dataset
}

predict_workout_plan(new_user_input)

✅ Model Loaded Successfully.

🔎 SEARCHING PLAN FOR: Muscle Gain | 4 Days | Home
✅ FOUND MATCH: User 45
   Match Goal : Weight Loss
   Match Freq : 5 Days
   Match Env  : Other
--------------------------------------------------

📅 Day 1 - Chest & Tris


,Day,Muscle Group,Exercise Name,Sets,Reps
0,Day 1 - Chest & Tris,Triceps,impossible dips,4,12-15
1,Day 1 - Chest & Tris,Abs,bicycle crunch,4,12-15
2,Day 1 - Chest & Tris,Triceps,impossible dips,4,12-15
3,Day 1 - Chest & Tris,Abs,bicycle crunch,4,12-15
4,Day 1 - Chest & Tris,Abs,bicycle crunch,4,12-15
5,Day 1 - Chest & Tris,Triceps,impossible dips,4,12-15
6,Day 1 - Chest & Tris,Triceps,impossible dips,4,12-15
7,Day 1 - Chest & Tris,Abs,bicycle crunch,4,12-15
8,Day 1 - Chest & Tris,Triceps,impossible dips,4,12-15
9,Day 1 - Chest & Tris,Abs,bicycle crunch,4,12-15



📅 Day 2 - Back & Bis


,Day,Muscle Group,Exercise Name,Sets,Reps
0,Day 2 - Back & Bis,Back,superman,4,12-15
1,Day 2 - Back & Bis,Back,superman,4,12-15
2,Day 2 - Back & Bis,Back,superman,4,12-15
3,Day 2 - Back & Bis,Cardio,Football,1,37 Mins
4,Day 2 - Back & Bis,Back,superman,4,12-15
5,Day 2 - Back & Bis,Cardio,Football,1,37 Mins
6,Day 2 - Back & Bis,Cardio,Football,1,37 Mins
7,Day 2 - Back & Bis,Back,superman,4,12-15
8,Day 2 - Back & Bis,Cardio,Football,1,37 Mins
9,Day 2 - Back & Bis,Cardio,Football,1,37 Mins



📅 Day 3 - Legs Quad Focus


,Day,Muscle Group,Exercise Name,Sets,Reps
0,Day 3 - Legs Quad Focus,Abs,bicycle crunch,4,12-15
1,Day 3 - Legs Quad Focus,Cardio,Football,1,39 Mins
2,Day 3 - Legs Quad Focus,Calves,standing calf raise,4,12-15
3,Day 3 - Legs Quad Focus,Abs,bicycle crunch,4,12-15
4,Day 3 - Legs Quad Focus,Calves,standing calf raise,4,12-15
...,...,...,...,...,...
60,Day 3 - Legs Quad Focus,Quads,reverse lunge,4,12-15
61,Day 3 - Legs Quad Focus,Abs,bicycle crunch,4,12-15
62,Day 3 - Legs Quad Focus,Cardio,Football,1,39 Mins
63,Day 3 - Legs Quad Focus,Quads,bodyweight squat,4,12-15



📅 Day 4 - Shoulders & Abs


,Day,Muscle Group,Exercise Name,Sets,Reps
0,Day 4 - Shoulders & Abs,Abs,bicycle crunch,4,12-15
1,Day 4 - Shoulders & Abs,Shoulders,pike push up,4,12-15
2,Day 4 - Shoulders & Abs,Cardio,Football,1,43 Mins
3,Day 4 - Shoulders & Abs,Cardio,Football,1,43 Mins
4,Day 4 - Shoulders & Abs,Shoulders,pike push up,4,12-15
5,Day 4 - Shoulders & Abs,Shoulders,pike push up,4,12-15
6,Day 4 - Shoulders & Abs,Shoulders,pike push up,4,12-15
7,Day 4 - Shoulders & Abs,Abs,bicycle crunch,4,12-15
8,Day 4 - Shoulders & Abs,Cardio,Football,1,43 Mins
9,Day 4 - Shoulders & Abs,Cardio,Football,1,43 Mins



📅 Day 5 - Legs Glute/Ham


,Day,Muscle Group,Exercise Name,Sets,Reps
0,Day 5 - Legs Glute/Ham,Hamstrings,bodyweight squat,4,12-15
1,Day 5 - Legs Glute/Ham,Hamstrings,glute bridge,4,12-15
2,Day 5 - Legs Glute/Ham,Hamstrings,bodyweight squat,4,12-15
3,Day 5 - Legs Glute/Ham,Glutes,reverse lunge,4,12-15
4,Day 5 - Legs Glute/Ham,Hamstrings,bodyweight squat,4,12-15
...,...,...,...,...,...
60,Day 5 - Legs Glute/Ham,Cardio,Football,1,31 Mins
61,Day 5 - Legs Glute/Ham,Cardio,Football,1,31 Mins
62,Day 5 - Legs Glute/Ham,Glutes,glute bridge,4,12-15
63,Day 5 - Legs Glute/Ham,Hamstrings,bodyweight squat,4,12-15
